# 03 — Hypothesis Testing

Formal case-vs-control comparisons for each analysis arm, as a check on the correlation
screen in `02` before committing to a regression specification.

- **Environmental:** Mann-Whitney U for each colonization-pressure variable (cases vs.
  controls); non-parametric since CP is right-skewed.
- **Patient:** Mann-Whitney U for antibiotic course counts; chi-square for `any_abx_0_60`.
- Benjamini-Hochberg correction across the many antibiotic-class / CP tests to control the
  false discovery rate.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

env = load_processed("environmental_mrsa.csv")
pat = load_processed("patient_mrsa.csv")

## Environmental: colonization pressure, cases vs. controls

In [ ]:
cp_cols = [c for c in env.columns if c.endswith("_cp")]

rows = []
cases = env.loc[env["group_binary"] == 1]
controls = env.loc[env["group_binary"] == 0]
for c in cp_cols:
    stat, p = mannwhitneyu(cases[c], controls[c], alternative="two-sided")
    rows.append({
        "variable": c,
        "median_case": cases[c].median(),
        "median_control": controls[c].median(),
        "p_value": p,
    })
env_results = pd.DataFrame(rows)
env_results["p_adj_bh"] = multipletests(env_results["p_value"], method="fdr_bh")[1]
env_results.sort_values("p_adj_bh")

## Patient: antibiotic exposure, cases vs. controls

In [ ]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]

rows = []
cases = pat.loc[pat["group_binary"] == 1]
controls = pat.loc[pat["group_binary"] == 0]
for c in abx_cols:
    stat, p = mannwhitneyu(cases[c], controls[c], alternative="two-sided")
    rows.append({
        "variable": c,
        "median_case": cases[c].median(),
        "median_control": controls[c].median(),
        "p_value": p,
    })
pat_results = pd.DataFrame(rows)
pat_results["p_adj_bh"] = multipletests(pat_results["p_value"], method="fdr_bh")[1]
pat_results.sort_values("p_adj_bh")

In [ ]:
ct = pd.crosstab(pat["group"], pat["any_abx_0_60"])
chi2, p, dof, expected = chi2_contingency(ct)
print(ct)
print(f"chi2={chi2:.2f}, p={p:.4g}")

## Summary

Carry the predictors significant after BH correction (from both arms) into the logistic
regression stage as the primary covariates of interest; non-significant ones are still
included if the literature supports them, but are not the focus of interpretation.